# Appendix A — Leftover Patterns

The GoF patterns not covered in the main chapters. Each gets a compact Python example and interview notes.

Patterns covered:
1. **Bridge** — Decouple abstraction from implementation
2. **Builder** — Construct complex objects step by step
3. **Chain of Responsibility** — Pass requests along a chain of handlers
4. **Flyweight** — Share fine-grained objects to reduce memory
5. **Interpreter** — Define a grammar and interpret sentences
6. **Mediator** — Define how objects interact via a central mediator
7. **Memento** — Capture and restore object state (undo)
8. **Prototype** — Clone existing objects
9. **Visitor** — Define new operations on elements without changing their classes

---
## 1. Bridge Pattern
**Intent:** Decouple an abstraction from its implementation so both can vary independently.

Key: abstraction holds a reference to the implementation (composition instead of inheritance).

In [ ]:
from abc import ABC, abstractmethod

# Implementation hierarchy
class Renderer(ABC):
    @abstractmethod
    def render_circle(self, radius: float): ...

class VectorRenderer(Renderer):
    def render_circle(self, radius): print(f"Drawing circle (vector) r={radius}")

class RasterRenderer(Renderer):
    def render_circle(self, radius): print(f"Drawing circle (raster) r={radius} px")


# Abstraction hierarchy
class Shape(ABC):
    def __init__(self, renderer: Renderer):
        self._renderer = renderer   # bridge

    @abstractmethod
    def draw(self): ...
    @abstractmethod
    def resize(self, factor: float): ...

class Circle(Shape):
    def __init__(self, renderer, radius):
        super().__init__(renderer)
        self._radius = radius

    def draw(self): self._renderer.render_circle(self._radius)
    def resize(self, factor): self._radius *= factor


c1 = Circle(VectorRenderer(), 5)
c2 = Circle(RasterRenderer(), 5)
c1.draw()
c2.draw()
c1.resize(2); c1.draw()  # resize doesn't affect renderer

---
## 2. Builder Pattern
**Intent:** Separate the construction of a complex object from its representation.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Pizza:
    size: str
    crust: str
    sauce: str
    toppings: list[str] = field(default_factory=list)
    extra_cheese: bool = False

    def __repr__(self):
        tops = ', '.join(self.toppings) or 'none'
        return (f"{self.size} pizza: {self.crust} crust, {self.sauce} sauce, "
                f"toppings=[{tops}], extra_cheese={self.extra_cheese}")


class PizzaBuilder:
    def __init__(self, size: str):
        self._size  = size
        self._crust = "thin"
        self._sauce = "tomato"
        self._toppings: list[str] = []
        self._extra_cheese = False

    def crust(self, c: str)       -> "PizzaBuilder": self._crust = c; return self
    def sauce(self, s: str)       -> "PizzaBuilder": self._sauce = s; return self
    def topping(self, t: str)     -> "PizzaBuilder": self._toppings.append(t); return self
    def extra_cheese(self)        -> "PizzaBuilder": self._extra_cheese = True; return self

    def build(self) -> Pizza:
        return Pizza(self._size, self._crust, self._sauce,
                     self._toppings, self._extra_cheese)


p = (PizzaBuilder("large")
     .crust("stuffed")
     .sauce("pesto")
     .topping("mushrooms")
     .topping("olives")
     .extra_cheese()
     .build())
print(p)

---
## 3. Chain of Responsibility
**Intent:** Pass a request along a chain of handlers; each decides to handle it or forward it.

In [ ]:
class SupportHandler(ABC):
    def __init__(self):
        self._next: Optional["SupportHandler"] = None

    def set_next(self, handler: "SupportHandler") -> "SupportHandler":
        self._next = handler
        return handler  # enables chaining: a.set_next(b).set_next(c)

    def handle(self, level: int, issue: str):
        if self._next:
            self._next.handle(level, issue)
        else:
            print(f"  [Unhandled] {issue}")


class L1Support(SupportHandler):
    def handle(self, level, issue):
        if level == 1:
            print(f"  [L1] Handling: {issue}")
        else:
            super().handle(level, issue)

class L2Support(SupportHandler):
    def handle(self, level, issue):
        if level == 2:
            print(f"  [L2] Handling: {issue}")
        else:
            super().handle(level, issue)

class L3Support(SupportHandler):
    def handle(self, level, issue):
        print(f"  [L3/Engineering] Handling: {issue} (severity={level})")


l1 = L1Support()
l2 = L2Support()
l3 = L3Support()
l1.set_next(l2).set_next(l3)

for lvl, issue in [(1, "Password reset"), (2, "Billing error"), (3, "Data corruption")]:
    l1.handle(lvl, issue)

---
## 4. Flyweight Pattern
**Intent:** Use sharing to efficiently support large numbers of fine-grained objects.

In [ ]:
import sys

class TreeType:  # Flyweight — shared intrinsic state
    def __init__(self, name: str, color: str, texture: str):
        self.name    = name
        self.color   = color
        self.texture = texture

    def draw(self, x, y):
        print(f"  {self.name}({self.color}) at ({x},{y})")


class TreeFactory:
    _cache: dict[tuple, TreeType] = {}

    @classmethod
    def get_type(cls, name, color, texture) -> TreeType:
        key = (name, color, texture)
        if key not in cls._cache:
            cls._cache[key] = TreeType(name, color, texture)
            print(f"  [Factory] Created new TreeType: {key}")
        return cls._cache[key]


class Tree:  # extrinsic state (position)
    def __init__(self, x, y, tree_type: TreeType):
        self.x, self.y = x, y
        self.tree_type = tree_type  # shared

    def draw(self): self.tree_type.draw(self.x, self.y)


oak = TreeFactory.get_type("Oak", "green", "rough")
trees = [Tree(i * 10, i * 5, oak) for i in range(5)]  # 5 trees, 1 shared type
TreeFactory.get_type("Oak", "green", "rough")  # no new object
for t in trees: t.draw()
print(f"\n  TreeType objects: {len(TreeFactory._cache)} (shared among {len(trees)} trees)")

---
## 5. Mediator Pattern
**Intent:** Define an object that encapsulates how a set of objects interact, promoting loose coupling.

In [ ]:
class ChatMediator:
    def __init__(self):
        self._users: list["ChatUser"] = []

    def add_user(self, u: "ChatUser"): self._users.append(u)

    def send(self, message: str, sender: "ChatUser"):
        for u in self._users:
            if u is not sender:
                u.receive(message, sender.name)


class ChatUser:
    def __init__(self, name: str, mediator: ChatMediator):
        self.name = name
        self._mediator = mediator
        mediator.add_user(self)

    def send(self, msg: str):
        print(f"{self.name}: {msg}")
        self._mediator.send(msg, self)

    def receive(self, msg: str, from_name: str):
        print(f"  [{self.name} receives from {from_name}]: {msg}")


chat = ChatMediator()
alice = ChatUser("Alice", chat)
bob   = ChatUser("Bob",   chat)
carol = ChatUser("Carol", chat)

alice.send("Hello everyone!")

---
## 6. Memento Pattern
**Intent:** Capture an object's internal state so it can be restored later, without violating encapsulation.

In [ ]:
from copy import deepcopy

class EditorMemento:
    def __init__(self, content: str, cursor: int):
        self._content = content
        self._cursor  = cursor

    @property
    def state(self): return (self._content, self._cursor)


class TextEditor:  # Originator
    def __init__(self):
        self._content = ""
        self._cursor  = 0

    def type(self, text: str):
        self._content = self._content[:self._cursor] + text + self._content[self._cursor:]
        self._cursor += len(text)

    def save(self) -> EditorMemento:
        return EditorMemento(self._content, self._cursor)

    def restore(self, m: EditorMemento):
        self._content, self._cursor = m.state

    def __repr__(self): return f"'{self._content}' (cursor={self._cursor})"


class History:  # Caretaker
    def __init__(self): self._stack: list[EditorMemento] = []
    def push(self, m): self._stack.append(m)
    def pop(self): return self._stack.pop() if self._stack else None


editor  = TextEditor()
history = History()

editor.type("Hello")
history.push(editor.save())
editor.type(", World")
history.push(editor.save())
editor.type("!!")
print("After edits:", editor)

editor.restore(history.pop())
print("After undo:", editor)
editor.restore(history.pop())
print("After undo:", editor)

---
## 7. Prototype Pattern
**Intent:** Create new objects by copying (cloning) an existing object.

In [ ]:
import copy

class GameCharacter:
    def __init__(self, name, hp, skills):
        self.name   = name
        self.hp     = hp
        self.skills = skills  # mutable list — deep copy needed

    def clone(self) -> "GameCharacter":
        return copy.deepcopy(self)  # Python's built-in clone

    def __repr__(self):
        return f"GameCharacter(name={self.name}, hp={self.hp}, skills={self.skills})"


template = GameCharacter("Warrior", 100, ["Slash", "Block"])
player1  = template.clone()
player1.name = "Alice"
player1.skills.append("Berserk")

player2 = template.clone()
player2.name = "Bob"

print("Template:", template)
print("Player 1:", player1)  # has Berserk
print("Player 2:", player2)  # clean copy

---
## 8. Interpreter Pattern
**Intent:** Given a language, define a representation for its grammar and an interpreter.

In [ ]:
# Simple boolean expression interpreter
class Expression(ABC):
    @abstractmethod
    def interpret(self, ctx: dict) -> bool: ...

class Variable(Expression):
    def __init__(self, name): self._name = name
    def interpret(self, ctx): return ctx.get(self._name, False)

class AndExpression(Expression):
    def __init__(self, left, right): self._l, self._r = left, right
    def interpret(self, ctx): return self._l.interpret(ctx) and self._r.interpret(ctx)

class OrExpression(Expression):
    def __init__(self, left, right): self._l, self._r = left, right
    def interpret(self, ctx): return self._l.interpret(ctx) or self._r.interpret(ctx)

class NotExpression(Expression):
    def __init__(self, expr): self._e = expr
    def interpret(self, ctx): return not self._e.interpret(ctx)


# Build: (is_admin OR is_owner) AND NOT is_banned
expr = AndExpression(
    OrExpression(Variable("is_admin"), Variable("is_owner")),
    NotExpression(Variable("is_banned"))
)

contexts = [
    {"is_admin": True,  "is_owner": False, "is_banned": False},
    {"is_admin": False, "is_owner": True,  "is_banned": True},
    {"is_admin": False, "is_owner": False, "is_banned": False},
]
for ctx in contexts:
    print(f"  {ctx} → {expr.interpret(ctx)}")

---
## 9. Visitor Pattern
**Intent:** Represent an operation to be performed on elements of an object structure.  
Visitor lets you define new operations without changing the element classes.

In [ ]:
class Visitor(ABC):
    @abstractmethod
    def visit_circle(self, c: "CircleShape"): ...
    @abstractmethod
    def visit_rectangle(self, r: "RectShape"): ...

class ShapeElement(ABC):
    @abstractmethod
    def accept(self, visitor: Visitor): ...

class CircleShape(ShapeElement):
    def __init__(self, radius): self.radius = radius
    def accept(self, v): v.visit_circle(self)

class RectShape(ShapeElement):
    def __init__(self, w, h): self.w, self.h = w, h
    def accept(self, v): v.visit_rectangle(self)


class AreaCalculator(Visitor):
    import math
    def visit_circle(self, c):
        import math
        print(f"  Circle area = {math.pi * c.radius**2:.2f}")
    def visit_rectangle(self, r):
        print(f"  Rectangle area = {r.w * r.h}")

class XMLExporter(Visitor):
    def visit_circle(self, c):
        print(f"  <circle radius=\"{c.radius}\"/>")
    def visit_rectangle(self, r):
        print(f"  <rectangle width=\"{r.w}\" height=\"{r.h}\"/>")


shapes: list[ShapeElement] = [
    CircleShape(5), RectShape(4, 6), CircleShape(3)
]

print("--- Area ---")
area = AreaCalculator()
for s in shapes: s.accept(area)

print("--- XML ---")
xml = XMLExporter()
for s in shapes: s.accept(xml)

---
## Quick Reference

| Pattern | Family | One-line intent | Python shortcut |
|---|---|---|---|
| Bridge | Structural | Separate abstraction from implementation | ABC + composition |
| Builder | Creational | Step-by-step complex object construction | Method chaining |
| Chain of Responsibility | Behavioral | Pass request along a handler chain | Linked nodes or list of handlers |
| Flyweight | Structural | Share fine-grained objects | Factory + dict cache; `__slots__` |
| Interpreter | Behavioral | Grammar + interpreter for a language | AST classes |
| Mediator | Behavioral | Central hub for object communication | Event bus / signal |
| Memento | Behavioral | Snapshot state for undo | `copy.deepcopy` |
| Prototype | Creational | Clone existing objects | `copy.deepcopy` |
| Visitor | Behavioral | New operations without changing elements | `accept/visit` double dispatch |